In [ ]:
# Path defined to accessing Brave (Used on VS Code)
BRAVE_PATH = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"  =
HEADLESS = True

# We are focusing on the last 25 years of players as NBA has changed a lot since the year 2000
SEASON_START = 2000
SEASON_END   = 2025

# Basketball Reference URL and the Output CSV files
BASE_URL = "https://www.basketball-reference.com"
PLAYER_IDS_CSV = "player_ids.csv"
OUT_CSV = "nba_player_stats.csv"

# Imports
import os
import time
import string
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup, Comment

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


In [ ]:
def get_driver():
    # Make browser settings (this tells Selenium how to open Brave)
    opts = Options()
    opts.binary_location = BRAVE_PATH  # Tell Selenium where Brave is installed

    # If I want it to run without showing the browser window
    if HEADLESS:
        opts.add_argument("--headless=new")  # Run in the background

    # These lines just make things run smoother and avoid some browser errors
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")

    # Set browser window size so pages look right
    opts.add_argument("--window-size=1920,1080")

    # Pretend to be a normal user (not a bot) by setting a common user agent
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    )

    # This automatically downloads and picks the right ChromeDriver version
    service = Service(ChromeDriverManager().install())

    # Finally, start up the browser with all those settings
    return webdriver.Chrome(service=service, options=opts)



In [ ]:
def _extract_table_from_soup(soup: BeautifulSoup, table_id: str):
    table = soup.find("table", id=table_id)
    if table:
        return pd.read_html(str(table))[0]

    # Basketball-Reference hides some tables in HTML comments so I look there too
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
        csoup = BeautifulSoup(comment, "html.parser")
        table = csoup.find("table", id=table_id)
        if table:
            return pd.read_html(str(table))[0]
    return None

def _season_start_year(season_str: str):
    try:
        return int(str(season_str)[:4])
    except Exception:
        return None

def _filter_seasons(df: pd.DataFrame | None):
    if df is None or df.empty or "Season" not in df.columns:
        return None
    out = df.copy()
    out["SeasonYear"] = out["Season"].apply(_season_start_year)
    out = out[out["SeasonYear"].between(SEASON_START, SEASON_END, inclusive="both")]
    if out.empty:
        return None
    return out.drop(columns=["SeasonYear"])

def _normalize_columns(df: pd.DataFrame):
    # I drop unnamed junk
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    rename_map = {"Tm": "Team", "Lg": "Lg", "Pos": "Pos", "Age": "Age"}
    present = {k: v for k, v in rename_map.items() if k in df.columns}
    return df.rename(columns=present)


In [ ]:
# I keep player ID scraping separate so I can refresh IDs without touching stats logic
def scrape_player_ids_for_letter(driver, letter: str):
    url = f"{BASE_URL}/players/{letter}/"
    driver.get(url)

    # Tiny sleep after to let layout settle (had issues with this initally)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "players")))
    time.sleep(0.4)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("table", id="players")
    if not table:
        return []

    ids = []
    for row in table.tbody.find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue
        link = row.find("a")
        if not link or "href" not in link.attrs:
            continue
        href = link["href"]  # /players/j/jamesle01.html as an example
        player_id = href.rsplit("/", 1)[-1].replace(".html", "")
        player_name = link.get_text(strip=True)
        ids.append((player_id, player_name))
    return ids

def save_player_ids(rows, path: str = PLAYER_IDS_CSV):
    df = pd.DataFrame(rows, columns=["PlayerID", "PlayerName"])
    df.to_csv(path, index=False)
    return len(df)

def run_scrape_ids():
    driver = get_driver()
    all_rows = []
    try:
        for letter in string.ascii_lowercase:
            print(f"→ {letter.upper()}: fetching…")
            rows = scrape_player_ids_for_letter(driver, letter)
            print(f"   found {len(rows)}")
            all_rows.extend(rows)
    finally:
        driver.quit()

    total = save_player_ids(all_rows, PLAYER_IDS_CSV)
    print(f"Saved {total} players to {Path(PLAYER_IDS_CSV).resolve()}")


In [ ]:
# I separate scraping from merging; makes it clear what's happening
def load_player_ids(path: str = PLAYER_IDS_CSV):
    if not Path(path).is_file():
        raise FileNotFoundError(f"{path} not found. Run run_scrape_ids() first.")
    df = pd.read_csv(path)
    return list(zip(df["PlayerID"], df["PlayerName"]))

def scrape_player_stats(driver, player_id: str):
    first_letter = player_id[0]
    url = f"{BASE_URL}/players/{first_letter}/{player_id}.html"
    driver.get(url)

    # Wait for any table; then I parse what I need
    WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.TAG_NAME, "table")))
    time.sleep(0.4)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    per_game = _extract_table_from_soup(soup, "per_game_stats")
    advanced = _extract_table_from_soup(soup, "advanced")

    per_game = _filter_seasons(per_game)
    advanced = _filter_seasons(advanced)

    if per_game is not None:
        per_game = _normalize_columns(per_game)
    if advanced is not None:
        advanced = _normalize_columns(advanced)

    return per_game, advanced

def _merge_player_frames(player_name: str, player_id: str,
                         per_game: pd.DataFrame | None,
                         advanced: pd.DataFrame | None) -> pd.DataFrame:
    if per_game is not None:
        per_game = per_game.copy()
        per_game["PlayerName"] = player_name
        per_game["PlayerID"] = player_id
    if advanced is not None:
        advanced = advanced.copy()
        advanced["PlayerName"] = player_name
        advanced["PlayerID"] = player_id

    if per_game is not None and advanced is not None:
        merged = pd.merge(
            per_game, advanced,
            on=["Season", "Team", "Lg"],
            suffixes=("_per_game", "_adv"),
            how="outer",
        )
        merged["PlayerName"] = player_name
        merged["PlayerID"] = player_id
        return merged

    if per_game is not None:
        return per_game
    if advanced is not None:
        return advanced

    return pd.DataFrame()

def _append_to_csv(df: pd.DataFrame, path: str = OUT_CSV):
    write_header = not Path(path).is_file()
    df.to_csv(path, index=False, mode="a", header=write_header)


In [ ]:
# I make the runner resume-safe so I can stop/start without duplicating rows

def run_scrape_stats():
    players = load_player_ids(PLAYER_IDS_CSV)

    scraped_ids = set()
    if Path(OUT_CSV).is_file():
        try:
            existing = pd.read_csv(OUT_CSV, usecols=["PlayerID"])
            scraped_ids = set(existing["PlayerID"].unique())
            print(f"Resuming… already in file: {len(scraped_ids)} players")
        except Exception:
            print("Warning: existing CSV unreadable for resume; starting fresh.")

    driver = get_driver()
    try:
        for idx, (pid, pname) in enumerate(players, start=1):
            if pid in scraped_ids:
                print(f"[{idx}/{len(players)}] skip {pname} ({pid})")
                continue

            print(f"[{idx}/{len(players)}] scrape {pname} ({pid})")
            per_game, advanced = scrape_player_stats(driver, pid)

            if per_game is None and advanced is None:
                print(f"   no seasons {SEASON_START}-{SEASON_END}; skip")
                continue

            merged = _merge_player_frames(pname, pid, per_game, advanced)
            _append_to_csv(merged, OUT_CSV)
            print(f"   appended {len(merged)} rows")
            time.sleep(1.2)
    finally:
        driver.quit()

    print(f"Done. Output → {Path(OUT_CSV).resolve()}")
